# Generate LLM text-generation training dataset

This notebook converts a dataframe containing records like:

```python
{
  "entry_data": {...},   # tabular attributes used to build the prompt
  "text": "..."          # assistant answer / target report
}
```

into chat-format JSONL files for supervised fine-tuning.


In [2]:
import json
import math
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
_rng = np.random.default_rng(SEED)


## 1. Prompt helpers

Transform `entry_data` into the clinical-data block, then wrap it in the report-generation prompt.


In [3]:
def normalize_text(value):
    if value is None:
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("utf-8")
    return value


def convert_cerb_to_her2_score_random(value, p_zero=0.7, rng=None):
    """Convert cerb_tumor_0 values to a HER2 score.

    Note: 'negatif' is randomly mapped to 0 or 1+.
    Keep SEED fixed for reproducibility.
    """
    v = normalize_text(value)
    rng = rng or _rng

    if v == "positif":
        return "3+"
    if v == "douteux":
        return "2+"
    if v == "negatif":
        return rng.choice(["0", "1+"], p=[p_zero, 1 - p_zero])
    if v in {"unknown", ""}:
        return "unknown"
    return "unknown"


def _safe_int(value):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return None


def create_donnees_cliniques(entry_data):
    """Build the tabular clinical-data block from entry_data only."""
    if not isinstance(entry_data, dict):
        raise TypeError(f"entry_data must be a dict, got {type(entry_data)}")

    in_situ = {
        "carcinome in situ",
        "carcinome intracanalaire non infiltrant",
        "carcinome lobulaire in situ",
        "carcinome canalaire in situ",
        "adenocarcinome papillaire intracanalaire non infiltrant",
        "adenocarcinome in situ",
        "carcinome intracanalaire et carcinome lobulaire in situ",
    }

    donnees_cliniques = ""

    type_diag = entry_data.get("type_diagnostique", "unknown")
    if type_diag != "unknown":
        donnees_cliniques += f"- Échantillon : {type_diag}\n"
    else:
        type_diag = "tumorectomie"
        if (
            entry_data.get("taille_tumor_0", "unknown") == "unknown"
            or (
                entry_data.get("ganglions_preleves", "unknown") == "unknown"
                and entry_data.get("taille_tumor_0", "unknown") == "0"
            )
        ):
            type_diag = "biopsie"
        donnees_cliniques += f"- Échantillon : {type_diag}\n"

    morpho_raw = entry_data.get("ref_morpho_name_tumor_0", "unknown")
    if morpho_raw != "unknown":
        morpho = (
            "carcinome canalaire in situ"
            if morpho_raw == "carcinome intracanalaire non infiltrant"
            else morpho_raw
        )
        donnees_cliniques += f"- Diagnostic (morphologie) : {morpho}\n"

    if morpho_raw in in_situ:
        donnees_cliniques += "- Grade SBR : non applicable (lésion non infiltrante)\n"
    elif entry_data.get("ref_grade_tumor_0", "unknown") == "unknown":
        donnees_cliniques += "- Grade SBR : non évalué\n"
    else:
        donnees_cliniques += f"- Grade SBR : {entry_data.get('ref_grade_tumor_0', '')}\n"

    taille = entry_data.get("taille_tumor_0", "unknown")
    if taille != "unknown" and type_diag != "biopsie":
        donnees_cliniques += f"- Taille tumorale : {taille} mm\n"
    elif taille == "unknown":
        if type_diag in {"biopsie", "curage ganglionnaire"}:
            donnees_cliniques += f"- Taille tumorale : non applicable ({type_diag})\n"
        else:
            donnees_cliniques += "- Taille tumorale : non évaluée\n"

    ganglions_preleves = entry_data.get("ganglions_preleves", "unknown")
    if ganglions_preleves != "unknown":
        donnees_cliniques += f"- Nombre de ganglions examinés : {ganglions_preleves}\n"
    else:
        donnees_cliniques += "- Nombre de ganglions examinés : non évalués\n"

    ganglions_atteints = entry_data.get("ganglions_atteints", "unknown")
    if ganglions_preleves != "unknown" and ganglions_atteints != "unknown":
        donnees_cliniques += (
            f"- Nombre de ganglions atteints : {ganglions_atteints} "
            f"({ganglions_atteints}/{ganglions_preleves})\n"
        )
    elif ganglions_preleves != "unknown" and ganglions_atteints == "unknown":
        donnees_cliniques += "- Nombre de ganglions atteints : non spécifié\n"

    emb = entry_data.get("embols_vasculaires_tumor_0", "unknown")
    if emb != "unknown":
        if emb == "1":
            donnees_cliniques += "- Présence d’emboles vasculaires\n"
        else:
            donnees_cliniques += "- Absence d’emboles vasculaires\n"
    else:
        donnees_cliniques += "- Emboles vasculaires : non évalués\n"

    rupture_caps = str(entry_data.get("rupture_capsulaire", "unknown")).strip().lower()
    gang_atteints_int = _safe_int(ganglions_atteints)
    has_positive_nodes = gang_atteints_int is not None and gang_atteints_int > 0

    if rupture_caps == "non":
        donnees_cliniques += "- Présence de rupture capsulaire : non\n"
    elif rupture_caps == "oui" and has_positive_nodes:
        donnees_cliniques += "- Présence de rupture capsulaire : oui\n"
    else:
        donnees_cliniques += "- Présence de rupture capsulaire : non évaluée\n"

    re = entry_data.get("re_tumor_0", "unknown")
    if re == "unknown":
        donnees_cliniques += "- RE ou RO (récepteurs aux œstrogènes) : non évalués\n"
    elif re == "positifs":
        donnees_cliniques += (
            f"- RE ou RO (récepteurs aux œstrogènes) : positifs ; "
            f"pourcentage : {entry_data['re_perc']}%\n"
        )
    else:
        donnees_cliniques += "- RE ou RO (récepteurs aux œstrogènes) : négatifs\n"

    rp = entry_data.get("rp_tumor_0", "unknown")
    if rp == "unknown":
        donnees_cliniques += "- RP (récepteurs à la progestérone) : non évalués\n"
    elif rp == "positifs":
        donnees_cliniques += (
            f"- RP (récepteurs à la progestérone) : positifs ; "
            f"pourcentage : {entry_data['rp_perc']}%\n"
        )
    else:
        donnees_cliniques += "- RP (récepteurs à la progestérone) : négatifs\n"

    cerb2 = convert_cerb_to_her2_score_random(
        entry_data.get("cerb_tumor_0", "unknown"),
        p_zero=0.7,
    )
    if cerb2 != "unknown":
        donnees_cliniques += f"- HER2 (CerbB2) : {cerb2}\n"
    else:
        donnees_cliniques += "- HER2 (CerbB2) : non évalué\n"

    ki67 = entry_data.get("ki67_tumor_0", "unknown")
    if ki67 != "unknown":
        donnees_cliniques += f"- Prolifération cellulaire (Ki67) : {ki67}%\n"
    else:
        donnees_cliniques += "- Prolifération cellulaire (Ki67) : non évaluée\n"

    marges = entry_data.get("marges_saines_tumor_0", "unknown")
    if marges != "unknown":
        donnees_cliniques += f"- Marges chirurgicales saines : {marges}\n"
    else:
        if type_diag == "biopsie":
            donnees_cliniques += "- Marges chirurgicales saines : non applicable (biopsie)\n"
        else:
            donnees_cliniques += "- Marges chirurgicales saines : non évaluées\n"

    return donnees_cliniques


In [4]:
SYSTEM_PROMPT = (
    "Tu es un médecin anatomopathologiste hospitalier expérimenté. "
    "Tu rédiges des comptes rendus anatomopathologiques complets, cohérents "
    "et conformes aux standards hospitaliers français. "
    "Tu corriges automatiquement toute incohérence médicale, stylistique ou logique."
)


def generate_report_prompt(donnees_cliniques, system_prompt=True):
    user_prompt = f"""Tache:
Rédige un **compte rendu anatomopathologique complet en français médical** à partir des données cliniques délimitées ### DONNÉES CLINIQUES ### ci-dessous.

### DONNÉES CLINIQUES ###

{donnees_cliniques}

### FIN DONNÉES CLINIQUES ###


Tu dois produire exactement les sections suivantes :
- Titre (indiquer une latéralité et un quadrant choisis de façon plausible)
- Examen macroscopique
- Examen microscopique
- Étude immunohistochimique
- Conclusion

────────────────────────────────────────
RÈGLES OBLIGATOIRES
────────────────────────────────────────
1) Cohérence générale
- Toute valeur fixe fournie doit être reprise exactement, sans modification.
- Ne jamais faire varier une même donnée entre les sections.

2) Données non évaluées
- Toute donnée indiquée comme "non évaluée", "non réalisée" ou "non disponible" dans les DONNÉES CLINIQUES doit être reprise strictement telle quelle.
- Ne jamais la compléter, estimer ou interpréter.
- Ne jamais la remplacer par une valeur normale ou pathologique (ex : "marges saines", "absence d’emboles", etc.).

3) Compléments descriptifs autorisés
- Tu peux enrichir la description macroscopique et microscopique avec des éléments plausibles, purement descriptifs et rédactionnels.
- Tu ne dois jamais modifier, compléter, interpréter ni contredire les données cliniques fournies.
- Les variables cliniques (taille tumorale, ganglions, marges, emboles, récepteurs, etc.) doivent être reprises strictement à l’identique.

4) Taille tumorale et pièce opératoire
- La taille tumorale ne doit jamais être inventée ou estimée si elle n’est pas fournie.
- Si elle est fournie, elle doit être identique dans la macroscopie, la microscopie et la conclusion.
- La taille tumorale doit toujours être inférieure ou égale à la taille de la pièce opératoire.
- Si poids et dimensions de la pièce sont ajoutés, ils doivent être plausibles et cohérents entre eux.

5) Hormonorécepteurs et Ki67
- Si RE ou RP sont positifs, utiliser le pourcentage fourni dans les données cliniques.
- Si RE ou RP sont négatifs, les rapporter comme négatifs sans inventer d’intensité absente.
- Une seule valeur par marqueur, sans contradiction.

6) Ganglions lymphatiques
- Le nombre de ganglions atteints doit toujours être inférieur ou égal au nombre de ganglions examinés.
- Le nombre décrit en macroscopie, microscopie et conclusion doit être identique.
- Si des ganglions sont examinés et qu’il n’y a aucune métastase ganglionnaire, indiquer 0/N dans la conclusion (N = nombre de ganglions examinés).
- Si le statut ganglionnaire est non évalué, ne pas mentionner de nombre de ganglions ni de statut (pas de 0/0, pN0, etc.).
- Ne pas mentionner de rupture capsulaire si les ganglions sont négatifs.

7) Grade
- Le grade SBR ne s’applique qu’aux lésions infiltrantes.
- Si un score chiffré est présent (ex. 3 + 2 + 1), calculer le total et en déduire le grade :
  - 3 à 5 : grade I
  - 6 à 7 : grade II
  - 8 à 9 : grade III
- Corriger toute incohérence entre score et grade.

────────────────────────────────────────
RÈGLES DE STYLE
────────────────────────────────────────
- Français médical clair, structuré, homogène, sans redondance.
- Fournir uniquement le rapport final.
- N’ajouter aucune explication, justification ou note hors compte rendu.
"""

    if system_prompt:
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

    inline_system_prompt = f"""RÔLE :
{SYSTEM_PROMPT}

"""
    return [{"role": "user", "content": f"{inline_system_prompt}{user_prompt}"}]



## 2. Load dataframe


In [5]:
df_selected_medgemma=pd.read_csv("../EvaluationOpenAI/LLMEvaluation/selectedMedGemma.csv")
df_selected_medgemmaRAG=pd.read_csv("../EvaluationOpenAI/LLMEvaluation/selectedMedGemma_RAG.csv")
df_selected_mixtral=pd.read_csv("../EvaluationOpenAI/LLMEvaluation/selectedMixtral.csv")
df_selected_mixtralRAG=pd.read_csv("../EvaluationOpenAI/LLMEvaluation/selectedMixtral_RAG.csv")
df_NoRAG = pd.concat([df_selected_medgemma, df_selected_mixtral], ignore_index=True)
df_RAG = pd.concat([df_selected_medgemmaRAG, df_selected_mixtralRAG], ignore_index=True)
df_all = pd.concat([df_selected_medgemma, df_selected_mixtral, df_selected_medgemmaRAG, df_selected_mixtralRAG], ignore_index=True)


In [6]:
df_all

,index,text,entry_data,source
0,1,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': {'value': 'biopsie', 're...",medGemma
1,2,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,{'type_diagnostique': {'value': 'tumorectomie'...,medGemma
2,3,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': {'value': 'mammectomie',...",medGemma
3,5,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': {'value': 'biopsie', 're...",medGemma
4,7,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': {'value': 'mammectomie',...",medGemma
...,...,...,...,...
2316,1042,SEIN GAUCHE : MAMMECTOMIE TOTALE\n\nExamen mac...,"{'type_diagnostique': {'value': 'mammectomie',...",medGemmaRAG
2317,1049,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': {'value': 'biopsie', 're...",medGemmaRAG
2318,1058,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Prél...,"{'type_diagnostique': {'value': 'biopsie', 're...",medGemmaRAG
2319,1059,SEIN GAUCHE : MAMMECTOMIE DU QUADRANT SUPÉRIEU...,"{'type_diagnostique': {'value': 'mammectomie',...",medGemmaRAG


## 3. Normalize the dataframe shape


In [7]:
import ast

def flatten_entry_data(entry_data):
    if isinstance(entry_data, str):
        entry_data = ast.literal_eval(entry_data)

    return {
        att: info.get("value", "unknown")
        for att, info in entry_data.items()
    }

    
def normalize_examples_dataframe(data):
    """Return a dataframe with at least columns: entry_data, text.

    Accepted inputs:
    - pd.DataFrame with entry_data/text columns
    - pd.DataFrame with eval column containing list(s) of records
    - dict with key "eval" containing records
    - list of records
    """
    if isinstance(data, dict) and "eval" in data:
        return pd.DataFrame(data["eval"])

    if isinstance(data, list):
        return pd.DataFrame(data)

    if not isinstance(data, pd.DataFrame):
        raise TypeError("data must be a DataFrame, dict with 'eval', or list of records")

    if {"entry_data", "text"}.issubset(data.columns):
        return data.copy()

    if "eval" in data.columns:
        records = []
        for value in data["eval"].dropna():
            if isinstance(value, list):
                records.extend(value)
            elif isinstance(value, dict):
                records.append(value)
            else:
                raise TypeError(f"Unexpected eval cell type: {type(value)}")
        return pd.DataFrame(records)

    raise ValueError("Could not find columns entry_data/text or an eval column.")



normalized_df = normalize_examples_dataframe(df_all)
normalized_df["entry_data"] = normalized_df["entry_data"].apply(flatten_entry_data)

normalized_df.head()


,index,text,entry_data,source
0,1,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': 'biopsie', 'ganglions_pr...",medGemma
1,2,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': 'tumorectomie', 'ganglio...",medGemma
2,3,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': 'mammectomie', 'ganglion...",medGemma
3,5,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': 'biopsie', 'ganglions_pr...",medGemma
4,7,**COMPTE RENDU ANATOMOPATHOLOGIQUE**\n\n**Titr...,"{'type_diagnostique': 'mammectomie', 'ganglion...",medGemma


<!-- ## 4. Build chat-format training examples -->

Each output item has this structure:

```json
{"messages": [
  {"role": "system", "content": "..."},
  {"role": "user", "content": "...prompt from entry_data only..."},
  {"role": "assistant", "content": "...text only..."}
]}
```


In [8]:
def make_training_example(row, system_prompt=True):
    entry_data = row["entry_data"]
    assistant_text = row["text"]

    if not isinstance(entry_data, dict):
        raise TypeError("row['entry_data'] must be a dict")
    if not isinstance(assistant_text, str) or not assistant_text.strip():
        raise ValueError("row['text'] must be a non-empty string")

    donnees_cliniques = create_donnees_cliniques(entry_data)
    messages = generate_report_prompt(donnees_cliniques, system_prompt=system_prompt)
    
    
    # Important: assistant target comes only from row["text"].
    messages.append({"role": "assistant", "content": assistant_text.strip()})
    # print(messages)
    return {"messages": messages}


def build_dataset(examples_df, system_prompt=True):
    required_cols = {"entry_data", "text"}
    missing = required_cols - set(examples_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    dataset = []
    skipped = []
    
    for i, row in examples_df.iterrows():
        
        try:
            dataset.append(make_training_example(row, system_prompt=system_prompt))
        except Exception as exc:
            print(row["entry_data"])
            skipped.append({"row_index": i, "error": str(exc)})

    return dataset, pd.DataFrame(skipped)


dataset, skipped_df = build_dataset(normalized_df, system_prompt=False)

print(f"Built {len(dataset)} training examples")
print(f"Skipped {len(skipped_df)} rows")

dataset[0]


Built 2321 training examples
Skipped 0 rows


{'messages': [{'role': 'user',
   'content': 'RÔLE :\nTu es un médecin anatomopathologiste hospitalier expérimenté. Tu rédiges des comptes rendus anatomopathologiques complets, cohérents et conformes aux standards hospitaliers français. Tu corriges automatiquement toute incohérence médicale, stylistique ou logique.\n\nTache:\nRédige un **compte rendu anatomopathologique complet en français médical** à partir des données cliniques délimitées ### DONNÉES CLINIQUES ### ci-dessous.\n\n### DONNÉES CLINIQUES ###\n\n- Échantillon : biopsie\n- Diagnostic (morphologie) : carcinome lobulaire\n- Grade SBR : SBR2\n- Taille tumorale : non applicable (biopsie)\n- Nombre de ganglions examinés : 1\n- Nombre de ganglions atteints : 1 (1/1)\n- Absence d’emboles vasculaires\n- Présence de rupture capsulaire : non évaluée\n- RE ou RO (récepteurs aux œstrogènes) : non évalués\n- RP (récepteurs à la progestérone) : positifs ; pourcentage : 30%\n- HER2 (CerbB2) : 1+\n- Prolifération cellulaire (Ki67) : 7%\n-

In [9]:
def getattr_llm(llm_att, df_entry_data, source):
    """
    Format df_entry_data into the same structure as the previous second return
    parameter, without extracting, selecting, or evaluating attributes.
    """

    data_all_attr = []

    for ind in df_entry_data.index:
        texte = llm_att["text"][ind] if "text" in llm_att else None
        entry_attrs = dict(df_entry_data.loc[ind])

        atts = {
            att: {
                "value": to_python_value(val),
                "reference": None,
            }
            for att, val in entry_attrs.items()
        }

        data_all_attr.append({
            "index": ind,
            "text": texte,
            "entry_data": atts,
            "source": source,
        })

    return pd.DataFrame(data_all_attr)

## 6. Train/validation split and JSONL export

In [16]:
def write_jsonl(records, path):
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return path

In [12]:
output_dir = Path("llm_dataset_output_all")
output_dir.mkdir(exist_ok=True)

# Adjust test_size as needed.
if len(dataset) >= 2:
    train_data, val_data = train_test_split(
        dataset,
        test_size=0.1,
        random_state=SEED,
        shuffle=True,
    )
else:
    train_data, val_data = dataset, []



train_path = write_jsonl(train_data, output_dir / "train.jsonl")
val_path = write_jsonl(val_data, output_dir / "validation.jsonl")
all_path = write_jsonl(dataset, output_dir / "all.jsonl")

print("Wrote:")
print(f"- {train_path} ({len(train_data)} examples)")
print(f"- {val_path} ({len(val_data)} examples)")
print(f"- {all_path} ({len(dataset)} examples)")


Wrote:
- llm_dataset_output_all/train.jsonl (2088 examples)
- llm_dataset_output_all/validation.jsonl (233 examples)
- llm_dataset_output_all/all.jsonl (2321 examples)


# NEW: Experiences: see impact of data Size

| Rank | Model           | Accuracy (%) | High + Perfect docs | Perfect docs |
| ---: | --------------- | -----------: | ------------------: | -----------: |
|    1 | **Mx_Filter**   |    **96.68** |             **998** |          674 |
|    2 | *MG_RAG_Filter* |      *96.66* |               *993* |      **689** |
|    3 | Mx_RAG_Filter   |        96.64 |                 990 |        *688* |
|    4 | MG_Filter       |        95.97 |                 973 |          605 |
|    5 | MG              |        94.63 |                 915 |          485 |
|    6 | MG_RAG          |        94.44 |                 900 |          490 |
|    7 | Mx              |        93.62 |                 827 |          489 |
|    8 | Mx_RAG          |        90.57 |                 666 |          355 |


### data sizes: 

In [ ]:
print("medgemma no rag")
dataset_MG_noFilter, skipped_MG_noFilter = build_dataset(normalized_MG_noFilter, system_prompt=False)

print(f"Built {len(dataset_MG_noFilter)} training examples")
print(f"Skipped {len(skipped_MG_noFilter)} rows")
print()

# write_dataset_splits(dataset_MxRag_Filter, "llm_dataset_output_MxRag_Filter")

print("medgemma filtered no rag")
dataset_MG_Filter, skipped_MG_Filter = build_dataset(normalized_MG_Filter, system_prompt=False)
print(f"Built {len(dataset_MG_Filter)} training examples")
print(f"Skipped {len(skipped_MG_Filter)} rows")
print()


print("medgemma rag")
dataset_MGrag_noFilter, skipped_MGrag_noFilter = build_dataset(normalized_MGrag_noFilter, system_prompt=False)
print(f"Built {len(dataset_MGrag_noFilter)} training examples")
print(f"Skipped {len(skipped_MGrag_noFilter)} rows")
print()


print("medgemma rag filtered")  
dataset_MGrag_Filter, skipped_MGrag_Filter = build_dataset(normalized_MGrag_Filter, system_prompt=False)
print(f"Built {len(dataset_MGrag_Filter)} training examples")
print(f"Skipped {len(skipped_MGrag_Filter)} rows")
print()

print("mixtral no rag")
dataset_Mx_noFilter, skipped_Mx_noFilter = build_dataset(normalized_Mx_noFilter, system_prompt=False)
print(f"Built {len(dataset_Mx_noFilter)} training examples")
print(f"Skipped {len(skipped_Mx_noFilter)} rows")
print()


print("mixtral filtered no rag")  #medgemma rag filtered 
# dataset_Mx_Filter, skipped_Mx_Filter = build_dataset(normalized_Mx_Filter, system_prompt=False)
dataset_Mx_Filter, skipped_Mx_Filter = build_dataset(normalized_MGrag_Filter, system_prompt=False)
print(f"Built {len(dataset_Mx_Filter)} training examples")
print(f"Skipped {len(skipped_Mx_Filter)} rows")
print()

print("mixtral rag")
dataset_MxRag_noFilter, skipped_MxRag_noFilter = build_dataset(normalized_MxRag_noFilter, system_prompt=False)
print(f"Built {len(dataset_MxRag_noFilter)} training examples")
print(f"Skipped {len(skipped_MxRag_noFilter)} rows")
print()


print("mixtral rag filtered")
dataset_MxRag_Filter, skipped_MxRag_Filter = build_dataset(normalized_MxRag_Filter, system_prompt=False)
print(f"Built {len(dataset_MxRag_Filter)} training examples")
print(f"Skipped {len(skipped_MxRag_Filter)} rows")
print()


## D_rk1 + D_rk2
Mx_Filter + MG_RAG_Filter

mixtral filtered no rag
(Built 561 training examples)
+
medgemma rag filtered
(Built 662 training examples)


In [26]:
# normalized_MGrag_Filter + normalized_Mx_Filter
df_rk_1_2 = pd.concat([df_selected_mixtral, df_selected_medgemmaRAG], ignore_index=True)
normalized_df_rk_1_2 = normalize_examples_dataframe(df_rk_1_2)
normalized_df_rk_1_2["entry_data"] = normalized_df_rk_1_2["entry_data"].apply(flatten_entry_data)

dataset_rk_1_2, skipped_rk_1_2 = build_dataset(normalized_df_rk_1_2, system_prompt=False)
print(f"Built {len(dataset_rk_1_2)} training examples")
print(f"Skipped {len(skipped_rk_1_2)} rows")
write_dataset_splits(dataset_rk_1_2, "llm_dataset_output_D_rk1_2")

Built 1223 training examples
Skipped 0 rows
Wrote:
- llm_dataset_output_D_rk1_2/train.jsonl (1100 examples)
- llm_dataset_output_D_rk1_2/validation.jsonl (123 examples)
- llm_dataset_output_D_rk1_2/all.jsonl (1223 examples)


(PosixPath('llm_dataset_output_D_rk1_2/train.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_2/validation.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_2/all.jsonl'))

## D_rk1 + D_rk2 + D_rk3
Mx_Filter + MG_RAG_Filter + Mx_RAG_Filter

mixtral filtered no rag
(Built 561 training examples)
+
medgemma rag filtered
(Built 662 training examples)
+
mixtral rag filtered
Built 408 training examples


In [27]:
# normalized_MGrag_Filter + normalized_Mx_Filter
df_rk_1_3 = pd.concat([df_selected_mixtral, df_selected_medgemmaRAG,df_selected_mixtralRAG], ignore_index=True)
normalized_df_rk_1_3 = normalize_examples_dataframe(df_rk_1_3)
normalized_df_rk_1_3["entry_data"] = normalized_df_rk_1_3["entry_data"].apply(flatten_entry_data)


dataset_rk_1_3, skipped_rk_1_3 = build_dataset(normalized_df_rk_1_3, system_prompt=False)
print(f"Built {len(dataset_rk_1_3)} training examples")
print(f"Skipped {len(skipped_rk_1_3)} rows")
write_dataset_splits(dataset_rk_1_3, "llm_dataset_output_D_rk1_3")

Built 1631 training examples
Skipped 0 rows
Wrote:
- llm_dataset_output_D_rk1_3/train.jsonl (1467 examples)
- llm_dataset_output_D_rk1_3/validation.jsonl (164 examples)
- llm_dataset_output_D_rk1_3/all.jsonl (1631 examples)


(PosixPath('llm_dataset_output_D_rk1_3/train.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_3/validation.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_3/all.jsonl'))

## D_rk1 + D_rk2 + D_rk3 + D_rk4
Mx_Filter + MG_RAG_Filter + Mx_RAG_Filter + MG_Filter

mixtral filtered no rag
(Built 561 training examples)
+
medgemma rag filtered
(Built 662 training examples)
+
mixtral rag filtered
Built 408 training examples
+
medgemma filtered no rag
Built 690 training examples


In [28]:
# normalized_MGrag_Filter + normalized_Mx_Filter
df_rk_1_4 = pd.concat([df_selected_mixtral, df_selected_medgemmaRAG,df_selected_mixtralRAG, df_selected_medgemma], ignore_index=True)
normalized_df_rk_1_4 = normalize_examples_dataframe(df_rk_1_4)
normalized_df_rk_1_4["entry_data"] = normalized_df_rk_1_4["entry_data"].apply(flatten_entry_data)

dataset_rk_1_4, skipped_rk_1_4 = build_dataset(normalized_df_rk_1_4, system_prompt=False)
print(f"Built {len(dataset_rk_1_4)} training examples")
print(f"Skipped {len(skipped_rk_1_4)} rows")
write_dataset_splits(dataset_rk_1_4, "llm_dataset_output_D_rk1_4")

Built 2321 training examples
Skipped 0 rows
Wrote:
- llm_dataset_output_D_rk1_4/train.jsonl (2088 examples)
- llm_dataset_output_D_rk1_4/validation.jsonl (233 examples)
- llm_dataset_output_D_rk1_4/all.jsonl (2321 examples)


(PosixPath('llm_dataset_output_D_rk1_4/train.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_4/validation.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_4/all.jsonl'))

## D_rk1 + D_rk2 + D_rk3 + D_rk4 + D_rk5 
Mx_Filter + MG_RAG_Filter + Mx_RAG_Filter + MG

mixtral filtered no rag
(Built 662 training examples)
+
medgemma rag filtered
(Built 561 training examples)
+
mixtral rag filtered
Built 408 training examples
+
medgemma no rag
Built 1062 training examples

(medgemma filtered no rag is included inside medgemma no rag)


In [29]:
# normalized_MGrag_Filter + normalized_Mx_Filter
df_rk_1_5 = pd.concat([df_selected_mixtral, df_selected_medgemmaRAG,df_selected_mixtralRAG, df_no_sel_medgemma], ignore_index=True)
normalized_df_rk_1_5 = normalize_examples_dataframe(df_rk_1_5)
normalized_df_rk_1_5["entry_data"] = normalized_df_rk_1_5["entry_data"].apply(flatten_entry_data)

dataset_rk_1_5, skipped_rk_1_5 = build_dataset(normalized_df_rk_1_5, system_prompt=False)
print(f"Built {len(dataset_rk_1_5)} training examples")
print(f"Skipped {len(skipped_rk_1_5)} rows")
write_dataset_splits(dataset_rk_1_5, "llm_dataset_output_D_rk1_5")

Built 2693 training examples
Skipped 0 rows
Wrote:
- llm_dataset_output_D_rk1_5/train.jsonl (2423 examples)
- llm_dataset_output_D_rk1_5/validation.jsonl (270 examples)
- llm_dataset_output_D_rk1_5/all.jsonl (2693 examples)


(PosixPath('llm_dataset_output_D_rk1_5/train.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_5/validation.jsonl'),
 PosixPath('llm_dataset_output_D_rk1_5/all.jsonl'))